# O'FAIRe scores of the EarthPortal ontologies

This notebook turns the O'FAIRe results into one table: **one row per ontology**, with its overall
scores and the points it earned on each of the 61 questions.

- **I.** Loading the snapshot
- **II.** Building the table
- **III.** Checking the table
- **IV.** Saving the table

In [1]:
# Imports
import json
import math

import pandas as pd

from config import *

# I. Loading the snapshot

In [2]:
# We load the catalogue (names, kinds) and the O'FAIRe results saved by notebook 01
catalogue = {o["acronym"]: o for o in json.load(open(RAW / "catalogue.json", encoding="utf-8"))}
ofaire = json.load(open(RAW / "ofaire.json", encoding="utf-8"))
print(f"Snapshot {RUN_DATE}: {len(catalogue)} entries, {len(ofaire)} O'FAIRe results")

Snapshot 2026-10-02: 92 entries, 92 O'FAIRe results


# II. Building the table

Two details matter here:

- Ten questions are **not implemented** in O'FAIRe: its code answers 0 with the message *"This question
  cannot be evaluated in the repository at this time"*, whatever the ontology and whatever the portal.
  That 0 does not mean the ontology failed, so we store those answers as empty (`NaN`).
- Two questions behave in a special way (see `reference/ofaire_known_issues.csv`): F3Q1 is always 0
  while F3Q2 and F3Q3 are given in full to every ontology, and R1.3Q2 only rewards AgroPortal groups or
  the OBO library. We keep the second one but count it as unreachable.

From this we compute a **reachable maximum**: the sum of the maximum of each question an EarthPortal
ontology can score on. It is 386 points: 478, minus 70 (the ten questions not implemented), minus 20
(R1.3Q2), minus 2 (the five questions of I1 add up to 42 points, but O'FAIRe counts 44 for I1).

In [3]:
known_issues = pd.read_csv(KNOWN_ISSUES)
known = dict(zip(known_issues.question, known_issues.treatment))
known_issues

,question,treatment,reason,evidence
0,F3Q1,not_evaluated,"The code gives 0 to every artefact, with the u...","F3.java line 25 (agroportal/fairness, commit 2..."
1,R1.3Q2,unreachable,The code gives credits only to groups whose na...,"R13.java lines 50 to 63 (agroportal/fairness, ..."


In [4]:
PRINCIPLES = ["Findable", "Accessible", "Interoperable", "Reusable"]
rows, questions = [], {}

for acronym, result in ofaire.items():
    row = {"acronym": acronym, "name": catalogue[acronym]["name"], "kind": kind_of(catalogue[acronym]),
           "normalized_score": result["normalizedScore"], "total_points": result["score"]}
    for principle in PRINCIPLES:
        for sub_principle, block in result[principle].items():
            if not isinstance(block, dict) or "results" not in block:
                continue                       # skip the principle's own score fields
            for qid, q in block["results"].items():
                not_implemented = "cannot be evaluated" in (q.get("explanation") or "")
                evaluable = not not_implemented and known.get(qid) != "not_evaluated"
                row[qid] = q["score"] if evaluable else math.nan
                questions.setdefault(qid, {"question": qid, "principle": principle,
                                           "sub_principle": sub_principle, "max_credits": q["maxCredits"],
                                           "evaluable": evaluable,
                                           "reachable": not not_implemented and known.get(qid) != "unreachable",
                                           "text": q["question"].strip()})
    rows.append(row)
questions = pd.DataFrame(questions.values())

# The reachable maximum: the sum of the maximum of each question an EarthPortal ontology can score on
reachable_max = int(questions[questions.evaluable & questions.reachable].max_credits.sum())
for row in rows:
    row["reachable_max"] = reachable_max
    row["reachable_score_pct"] = round(100 * row["total_points"] / reachable_max, 1)

# We put the overall columns first, then the 61 questions in O'FAIRe's own order
front = ["acronym", "name", "kind", "normalized_score", "total_points", "reachable_max", "reachable_score_pct"]
scores = pd.DataFrame(rows)[front + list(questions.question)]
scores = scores.sort_values(["kind", "reachable_score_pct"], ascending=[True, False]).reset_index(drop=True)
scores.head()

,acronym,name,kind,normalized_score,total_points,reachable_max,reachable_score_pct,F1Q1,F1Q2,F1Q3,...,R1.2Q2,R1.2Q3,R1.2Q4,R1.2Q5,R1.2Q6,R1.2Q7,R1.2Q8,R1.3Q1,R1.3Q2,R1.3Q3
0,OSO,Observatories of the Seas Ontology,artefact,75,362.36,386,93.9,9.0,11.0,12.0,...,4.0,6.0,4.0,NaN,6.0,4.0,2.0,10.0,0.0,6.0
1,NVS_OD1,ODATIS aggregation parameters and Essential Va...,artefact,61,292.75,386,75.8,9.0,0.0,12.0,...,2.0,2.0,4.0,NaN,2.0,2.0,2.0,10.0,0.0,6.0
2,EASY_DATA_VOCAB,EaSy Data Unified Thesaurus,artefact,60,288.00,386,74.6,9.0,0.0,12.0,...,4.0,4.0,4.0,NaN,4.0,2.0,2.0,10.0,0.0,6.0
3,NVS_P06,Governance for vocabularies used within the da...,artefact,60,287.75,386,74.5,9.0,0.0,12.0,...,0.0,2.0,4.0,NaN,2.0,2.0,0.0,5.0,0.0,6.0
4,NVS_P07,Climate and Forecast Standard Names,artefact,60,287.75,386,74.5,9.0,0.0,12.0,...,2.0,2.0,4.0,NaN,6.0,2.0,0.0,5.0,0.0,6.0


# III. Checking the table

In [5]:
# We check that the question scores add up to O'FAIRe's total, for every ontology
question_columns = list(questions.question)
if (scores[question_columns].sum(axis=1) - scores.total_points).abs().max() < 1e-6:
    print("For every ontology, the question scores add up exactly to O'FAIRe's total.")
else:
    print("Warning: for some ontologies, the question scores do not add up to O'FAIRe's total.")

# We check how O'FAIRe normalises: total / 478 * 100, rounded down
if (scores.normalized_score == (100 * scores.total_points / 478).apply(math.floor)).all():
    print("O'FAIRe's normalised score is the total out of 478, rounded down.")

# We check the reachable maximum against the criteria: 478, minus the questions without a usable
# answer or unreachable, minus the 2 points of I1 that no question gives
excluded = questions[~(questions.evaluable & questions.reachable) & (questions.question != "F3Q1")].max_credits.sum()
print(f"Questions an EarthPortal ontology can score on: {(questions.evaluable & questions.reachable).sum()} of {len(questions)}; "
      f"reachable maximum: {reachable_max} points = 478 - {excluded} - 2")
scores.kind.value_counts()

For every ontology, the question scores add up exactly to O'FAIRe's total.
O'FAIRe's normalised score is the total out of 478, rounded down.
Questions an EarthPortal ontology can score on: 49 of 61; reachable maximum: 386 points = 478 - 90.0 - 2


kind
artefact         71
view             16
metadata-only     5
Name: count, dtype: int64

**Conclusion**: the table is consistent with O'FAIRe. Artefacts, views and metadata-only records are
kept apart in the `kind` column; the analysis focuses on the artefacts.

# IV. Saving the table

In [6]:
scores.to_csv(DATA / f"ofaire_scores_{RUN_DATE}.csv", index=False)
questions.to_csv(DATA / f"ofaire_questions_{RUN_DATE}.csv", index=False)
print(f"Saved the two tables in {DATA.relative_to(REPO)}/")

Saved the two tables in earthportal_sa_audit/data/
